# 🐄 Taurus Vision — Real-time Stream Processor

## Arxitektura:
```
Webcam/Kamera
     ↓
Laptop (faqat JPEG yuborish — CPU minimal)
     ↓  HTTP POST (JPEG frames)
Colab GPU ← bu notebook
  ├── YOLO (sigir topish)
  ├── best.pt (muzzle topish)
  ├── MobileNetV2 (embedding)
  ├── Tracker
  └── push-tracks → Backend
```

## Tartib:
1. **Colab da**: Cell 1-5 ni ishga tushiring → ngrok URL oling
2. **Laptopda**: `laptop_stream_sender.py` ni ishga tushiring
3. **Natija**: LiveFeed da ko'rish mumkin


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — GPU + kutubxonalar
# ═══════════════════════════════════════════════════════════════════
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f'✅ GPU: {r.stdout.strip()}')
else:
    print('❌ GPU topilmadi! Runtime → Change runtime type → T4 GPU')

import subprocess
subprocess.run(['pip', 'install', '-q', 'ultralytics', 'scipy', 'flask', 'pyngrok'], check=True)
print('✅ Kutubxonalar tayyor')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — SOZLAMALAR
# ═══════════════════════════════════════════════════════════════════

# Backend URL (ngrok tunnel)
BACKEND_URL   = 'https://YOUR-NGROK-URL.ngrok-free.app'  # ← O'ZGARTIRING
COLAB_SECRET  = 'taurus123'
CAMERA_ID     = 'WEBCAM-01'

# Stream qabul qilish port (Colab da Flask server)
RECEIVER_PORT = 5050

# ngrok authtoken (https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN   = 'YOUR_NGROK_TOKEN'  # ← O'ZGARTIRING

# Identifikatsiya chegarasi
ID_THRESHOLD  = 0.70

# Har nechta kadrda bir DB ga yuborish
PUSH_EVERY_N  = 5

# Kadrni necha marta kichraytirish (1 = original, 2 = yarim)
# Tarmoq uchun: 2 tavsiya (tezroq, sifat yetarli)
FRAME_SCALE   = 1

# Backend ulanishini tekshirish
if 'YOUR-NGROK-URL' in BACKEND_URL:
    print('⚠️  BACKEND_URL hali o\'zgartirilmagan')
else:
    import requests
    try:
        h = {'ngrok-skip-browser-warning': '1'}
        if COLAB_SECRET: h['X-Colab-Key'] = COLAB_SECRET
        r = requests.get(f'{BACKEND_URL}/health/ready', headers=h, timeout=10)
        print(f'✅ Backend ulandi: {r.status_code}')
    except Exception as e:
        print(f'❌ Backend bilan aloqa yo\'q: {e}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Modellarni yuklash
# ═══════════════════════════════════════════════════════════════════
import torch, os
import torchvision.models as tvm
import torchvision.transforms as T
from ultralytics import YOLO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# YOLO — sigir topish
yolo_model = YOLO('yolo11n.pt').to(DEVICE)
print('✅ YOLO yuklandi')

# Muzzle detector — best.pt
if not os.path.exists('/content/best.pt'):
    from google.colab import files
    print('best.pt yuklang:')
    files.upload()
muzzle_model = YOLO('best.pt').to(DEVICE)
print('✅ Muzzle detector yuklandi')

# MobileNetV2 — backend bilan bir xil arxitektura!
_base = tvm.mobilenet_v2(weights=tvm.MobileNet_V2_Weights.IMAGENET1K_V1)
mobilenet = torch.nn.Sequential(
    _base.features,
    torch.nn.AdaptiveAvgPool2d((1, 1)),
    torch.nn.Flatten(),
)
for p in mobilenet.parameters():
    p.requires_grad = False
mobilenet.eval().to(DEVICE)

transform = T.Compose([
    T.ToPILImage(), T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print('✅ MobileNetV2 (ImageNet pretrained) yuklandi')
print(f'\nBarcha modellar {DEVICE.upper()} da ishlamoqda')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Embeddinglarni yuklash
# ═══════════════════════════════════════════════════════════════════
import requests, io, numpy as np

HEADERS = {'ngrok-skip-browser-warning': '1'}
if COLAB_SECRET:
    HEADERS['X-Colab-Key'] = COLAB_SECRET

print('Embeddinglar yuklanmoqda...')
r = requests.get(f'{BACKEND_URL}/api/v1/colab/export-embeddings',
                 headers=HEADERS, timeout=30)

if r.status_code != 200:
    print(f'❌ {r.status_code}: {r.text[:200]}')
    AVG_EMBEDDINGS = np.zeros((0, 1280), dtype=np.float32)
    AVG_ANIMAL_IDS = np.array([])
    AVG_TAG_IDS = []
else:
    data = np.load(io.BytesIO(r.content))
    EMBEDDINGS  = data['embeddings']
    ANIMAL_IDS  = data['animal_ids']
    TAG_IDS_RAW = data['tag_ids']

    unique_ids     = np.unique(ANIMAL_IDS)
    AVG_EMBEDDINGS = np.zeros((len(unique_ids), EMBEDDINGS.shape[1]), dtype=np.float32)
    AVG_ANIMAL_IDS = []
    AVG_TAG_IDS    = []
    for i, aid in enumerate(unique_ids):
        mask = ANIMAL_IDS == aid
        v = EMBEDDINGS[mask].mean(axis=0)
        v /= (np.linalg.norm(v) + 1e-8)
        AVG_EMBEDDINGS[i] = v
        AVG_ANIMAL_IDS.append(int(aid))
        AVG_TAG_IDS.append(str(TAG_IDS_RAW[mask][0]))
    AVG_ANIMAL_IDS = np.array(AVG_ANIMAL_IDS)

    print(f'✅ {len(unique_ids)} jonivor, {len(EMBEDDINGS)} embedding ({EMBEDDINGS.shape[1]}-dim)')
    print(f'   Teglar: {AVG_TAG_IDS[:8]}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Pipeline funksiyalari + Tracker
# ═══════════════════════════════════════════════════════════════════
import cv2, numpy as np, torch
from scipy.optimize import linear_sum_assignment

# ── Embedding ───────────────────────────────────────────────────────
def get_embedding(crop_bgr):
    if crop_bgr is None or crop_bgr.size == 0:
        return None
    try:
        rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        inp = transform(rgb).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            emb = mobilenet(inp).squeeze().cpu().numpy().astype(np.float32)
        emb /= (np.linalg.norm(emb) + 1e-8)
        return emb
    except:
        return None

def identify(emb):
    if emb is None or len(AVG_EMBEDDINGS) == 0:
        return None, 0.0
    sims   = AVG_EMBEDDINGS @ emb
    best_i = int(np.argmax(sims))
    score  = float(sims[best_i])
    return (int(AVG_ANIMAL_IDS[best_i]), score) if score >= ID_THRESHOLD else (None, score)

def get_muzzle_crop(frame, bbox_norm):
    H, W = frame.shape[:2]
    cx, cy, bw, bh = bbox_norm
    x1 = max(0, int((cx - bw/2) * W))
    y1 = max(0, int((cy - bh/2) * H))
    x2 = min(W, int((cx + bw/2) * W))
    y2 = min(H, int((cy + bh/2) * H))
    body = frame[y1:y2, x1:x2]
    if body.size < 200:
        return None
    mres = muzzle_model(body, verbose=False, conf=0.25)
    if mres and len(mres[0].boxes) > 0:
        mb = mres[0].boxes.xywhn[0].cpu().numpy()
        mh, mw = body.shape[:2]
        mx1 = max(0, int((mb[0]-mb[2]/2)*mw))
        my1 = max(0, int((mb[1]-mb[3]/2)*mh))
        mx2 = min(mw, int((mb[0]+mb[2]/2)*mw))
        my2 = min(mh, int((mb[1]+mb[3]/2)*mh))
        crop = body[my1:my2, mx1:mx2]
        if crop.size > 200:
            return crop
    return None  # muzzle topilmasa None

# ── Tracker ─────────────────────────────────────────────────────────
class Track:
    _n = 0
    def __init__(self, bbox, conf):
        Track._n += 1
        self.id = Track._n
        self.bbox = bbox
        self.conf = conf
        self.animal_id = None
        self.tag_id = None
        self.id_score = 0.0
        self.hits = 1
        self.age = 0
        self.id_attempts = 0
        self.state = 'tentative'
        self.last_emb = None
        self.vel_cx = 0.0
        self.vel_cy = 0.0

    def predict(self):
        cx, cy, w, h = self.bbox
        return (cx + self.vel_cx * self.age, cy + self.vel_cy * self.age, w, h)

    def update_vel(self, new_bbox):
        a = 0.4
        self.vel_cx = a*(new_bbox[0]-self.bbox[0]) + (1-a)*self.vel_cx
        self.vel_cy = a*(new_bbox[1]-self.bbox[1]) + (1-a)*self.vel_cy

def iou(a, b):
    ax,ay,aw,ah = a; bx,by,bw,bh = b
    xi = max(ax-aw/2, bx-bw/2); yi = max(ay-ah/2, by-bh/2)
    xa = min(ax+aw/2, bx+bw/2); ya = min(ay+ah/2, by+bh/2)
    inter = max(0,xa-xi)*max(0,ya-yi)
    return inter/(aw*ah+bw*bh-inter+1e-8)

tracks = []
IOU_THR   = 0.15
MAX_AGE   = 45
REID_THR  = 0.75
ID_INTV   = 6
MAX_ATT   = 30

def step_tracker(dets, frame, fno):
    global tracks
    matched_t, matched_d = set(), set()

    if tracks and dets:
        preds = [t.predict() for t in tracks]
        M = np.array([[iou(p, d[0]) for d in dets] for p in preds])
        ri, ci = linear_sum_assignment(-M)
        for r, c in zip(ri, ci):
            if M[r, c] >= IOU_THR:
                t = tracks[r]
                t.update_vel(dets[c][0])
                t.bbox = dets[c][0]; t.conf = dets[c][1]
                t.hits += 1; t.age = 0
                matched_t.add(r); matched_d.add(c)

    for i, t in enumerate(tracks):
        if i not in matched_t: t.age += 1

    for j, d in enumerate(dets):
        if j in matched_d: continue
        crop = get_muzzle_crop(frame, d[0])
        new_emb = get_embedding(crop) if crop is not None else None
        reid_done = False
        if new_emb is not None:
            best_s, best_i = REID_THR, -1
            for i, t in enumerate(tracks):
                if i in matched_t or t.last_emb is None or t.state == 'tentative': continue
                sim = float(np.dot(t.last_emb, new_emb))
                if sim > best_s: best_s, best_i = sim, i
            if best_i >= 0:
                t = tracks[best_i]
                t.update_vel(d[0]); t.bbox = d[0]; t.conf = d[1]
                t.hits += 1; t.age = 0; t.last_emb = new_emb
                matched_t.add(best_i); reid_done = True
        if not reid_done:
            nt = Track(d[0], d[1])
            nt.last_emb = new_emb
            tracks.append(nt)

    for t in tracks:
        if t.state == 'tentative' and t.hits >= 3:
            t.state = 'unidentified'

    for t in tracks:
        if t.state != 'unidentified' or t.animal_id or t.id_attempts >= MAX_ATT: continue
        if fno % ID_INTV != 0: continue
        t.id_attempts += 1
        crop = get_muzzle_crop(frame, t.bbox)
        if crop is None: continue
        emb = get_embedding(crop)
        if emb is not None:
            t.last_emb = emb
            aid, sc = identify(emb)
            if aid:
                t.animal_id = aid; t.id_score = sc; t.state = 'identified'
                t.tag_id = AVG_TAG_IDS[AVG_ANIMAL_IDS.tolist().index(aid)]

    tracks = [t for t in tracks if t.age < MAX_AGE]
    return tracks

print('✅ Pipeline funksiyalari tayyor')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Flask server + ngrok (Colab da stream qabul qilish)
# ═══════════════════════════════════════════════════════════════════
import threading, time, requests, json
import numpy as np
import cv2
from flask import Flask, request, jsonify
from pyngrok import ngrok, conf

# ngrok token
conf.get_default().auth_token = NGROK_TOKEN

app = Flask(__name__)

# Global state
state = {
    'fno':         0,
    'batch':       [],
    'fps':         0.0,
    'identified':  0,
    'total':       0,
    'last_push':   time.time(),
    't_last':      time.time(),
}

@app.route('/frame', methods=['POST'])
def receive_frame():
    """Laptopdan JPEG kadr qabul qilish va GPU da qayta ishlash."""
    try:
        # JPEG bytes → numpy frame
        jpg_bytes = request.data
        if not jpg_bytes:
            return jsonify({'error': 'empty'}), 400

        nparr = np.frombuffer(jpg_bytes, np.uint8)
        frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        if frame is None:
            return jsonify({'error': 'decode failed'}), 400

        # FRAME_SCALE bo'yicha kichraytirish
        if FRAME_SCALE > 1:
            h, w = frame.shape[:2]
            frame = cv2.resize(frame, (w//FRAME_SCALE, h//FRAME_SCALE))

        state['fno'] += 1
        fno = state['fno']

        # FPS hisoblash
        now = time.time()
        dt = now - state['t_last']
        state['fps'] = round(0.9 * state['fps'] + 0.1 * (1.0 / max(dt, 0.001)), 1)
        state['t_last'] = now

        # GPU da qayta ishlash
        results = yolo_model(frame, classes=[19], verbose=False, conf=0.35)
        dets = []
        if results and len(results[0].boxes) > 0:
            for box in results[0].boxes:
                b = box.xywhn[0].cpu().numpy()
                dets.append(((b[0], b[1], b[2], b[3]), float(box.conf[0])))

        active = step_tracker(dets, frame, fno)
        state['total'] = len(active)
        state['identified'] = sum(1 for t in active if t.state == 'identified')

        # Batch to'plash
        for t in active:
            if t.state == 'tentative': continue
            state['batch'].append({
                'track_id':     t.id,
                'animal_id':    t.animal_id,
                'tag_id':       t.tag_id,
                'state':        t.state,
                'confidence':   round(t.conf, 4),
                'id_score':     round(t.id_score, 4),
                'bbox':         {'x': round(t.bbox[0],4), 'y': round(t.bbox[1],4),
                                 'w': round(t.bbox[2],4), 'h': round(t.bbox[3],4)},
                'frame_number': fno,
                'video_time_s': round(fno / max(state['fps'], 1), 2),
            })

        # Har PUSH_EVERY_N kadrda backend ga yuborish
        if fno % PUSH_EVERY_N == 0 and state['batch']:
            payload = {
                'camera_id':    CAMERA_ID,
                'video_file':   None,
                'inference_ms': 0,
                'tracks':       state['batch'],
                'timestamp':    time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            }
            try:
                requests.post(
                    f'{BACKEND_URL}/api/v1/colab/push-tracks',
                    json=payload, headers=HEADERS, timeout=3,
                )
            except:
                pass
            state['batch'] = []

        return jsonify({
            'fno':        fno,
            'fps':        state['fps'],
            'tracks':     state['total'],
            'identified': state['identified'],
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/reload-embeddings', methods=['POST'])
def reload_embeddings():
    """Embeddinglarni qayta yuklash (yangi jonivor ro'yxatdan o'tganda)."""
    global AVG_EMBEDDINGS, AVG_ANIMAL_IDS, AVG_TAG_IDS
    try:
        r = requests.get(f'{BACKEND_URL}/api/v1/colab/export-embeddings',
                         headers=HEADERS, timeout=15)
        if r.status_code == 200:
            import io
            data = np.load(io.BytesIO(r.content))
            EMBS = data['embeddings']; AIDS = data['animal_ids']; TIDS = data['tag_ids']
            unique_ids = np.unique(AIDS)
            AVG_EMBEDDINGS = np.zeros((len(unique_ids), EMBS.shape[1]), dtype=np.float32)
            AVG_ANIMAL_IDS_list = []; AVG_TAG_IDS_list = []
            for i, aid in enumerate(unique_ids):
                mask = AIDS == aid
                v = EMBS[mask].mean(axis=0); v /= (np.linalg.norm(v)+1e-8)
                AVG_EMBEDDINGS[i] = v
                AVG_ANIMAL_IDS_list.append(int(aid))
                AVG_TAG_IDS_list.append(str(TIDS[mask][0]))
            AVG_ANIMAL_IDS = np.array(AVG_ANIMAL_IDS_list)
            AVG_TAG_IDS = AVG_TAG_IDS_list
            return jsonify({'ok': True, 'animals': len(unique_ids)})
    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500


@app.route('/status', methods=['GET'])
def status():
    return jsonify({
        'ok':         True,
        'fno':        state['fno'],
        'fps':        state['fps'],
        'tracks':     state['total'],
        'identified': state['identified'],
        'embeddings': len(AVG_ANIMAL_IDS),
        'device':     DEVICE,
    })


# Flask ni alohida threadda ishga tushirish
server_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=RECEIVER_PORT, debug=False, use_reloader=False),
    daemon=True,
)
server_thread.start()
time.sleep(2)

# ngrok tunnel ochish
tunnel = ngrok.connect(RECEIVER_PORT, 'http')
COLAB_STREAM_URL = tunnel.public_url

print(f'\n✅ Stream server tayyor!')
print(f'   Colab URL: {COLAB_STREAM_URL}')
print(f'\n📋 Laptop da quyidagi buyruqni ishga tushiring:')
print(f'   python laptop_stream_sender.py --url {COLAB_STREAM_URL}')
print(f'\n   Status tekshirish: {COLAB_STREAM_URL}/status')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Monitoring (ixtiyoriy — stream holati)
# ═══════════════════════════════════════════════════════════════════
import time
from IPython.display import clear_output

print('Monitoring ishlamoqda... (to\'xtatish uchun: Interrupt kernel)')
print(f'Stream URL: {COLAB_STREAM_URL}')

while True:
    clear_output(wait=True)
    print(f'=== Taurus Vision — Stream Monitor ===')
    print(f'Colab URL : {COLAB_STREAM_URL}')
    print(f'Kadrlar   : {state["fno"]}')
    print(f'FPS       : {state["fps"]}')
    print(f'Tracklar  : {state["total"]}')
    print(f'Aniqlandi : {state["identified"]}')
    print(f'Embeddinglar: {len(AVG_ANIMAL_IDS)} jonivor')
    print(f'Device    : {DEVICE.upper()}')
    print(f'\nYangi jonivor qo\'shilsa: POST {COLAB_STREAM_URL}/reload-embeddings')
    time.sleep(3)